In [84]:
import pandas as pd

df = pd.read_csv(r"../../data/final/Load_weather_history.csv")
df.head()

df = df[df["zone_id"] == 1]
df.drop(columns=["zone_id"], inplace=True)
df.sort_values(by="time", inplace=True)
df.reset_index(drop=True, inplace=True)
df["time"] = pd.to_datetime(df["time"])
df.head()

,time,load_kw,day_of_week,station_1_temp_c,station_2_temp_c,station_3_temp_c,station_4_temp_c,station_5_temp_c,station_6_temp_c,station_7_temp_c,station_8_temp_c,station_9_temp_c,station_10_temp_c,station_11_temp_c
0,2004-01-01 00:00:00,16853.0,Thursday,7.777778,3.333333,6.666667,7.222222,5.555556,6.666667,7.222222,6.111111,5.000000,5.555556,2.222222
1,2004-01-01 01:00:00,16450.0,Thursday,7.777778,2.222222,5.555556,6.111111,5.555556,6.111111,6.666667,6.666667,3.888889,6.111111,0.000000
2,2004-01-01 02:00:00,16517.0,Thursday,7.222222,1.666667,4.444444,5.000000,4.444444,5.555556,5.000000,5.555556,2.222222,6.111111,-0.555556
3,2004-01-01 03:00:00,16873.0,Thursday,5.000000,-1.111111,2.222222,2.777778,3.888889,3.333333,4.444444,1.111111,1.666667,3.888889,-1.111111
4,2004-01-01 04:00:00,17064.0,Thursday,3.888889,-1.111111,1.111111,0.555556,4.444444,3.333333,1.666667,-1.111111,0.555556,1.666667,1.111111


## Feature Engineering

- Create a hour column
- create dummies for the day of the week
- create dummies for the month


In [ ]:
df["hour"] = df["time"].astype("datetime64[ns]").dt.hour
df["hour"]
df["month"] = df["time"].astype("datetime64[ns]").dt.month


df = pd.get_dummies(df, columns=["day_of_week"], prefix="day")
df = pd.get_dummies(df, columns=["month"], prefix="month")
df = pd.get_dummies(df, columns=["hour"], prefix="hour")

df.head()

,time,load_kw,station_1_temp_c,station_2_temp_c,station_3_temp_c,station_4_temp_c,station_5_temp_c,station_6_temp_c,station_7_temp_c,station_8_temp_c,...,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,2004-01-01 00:00:00,16853.0,7.777778,3.333333,6.666667,7.222222,5.555556,6.666667,7.222222,6.111111,...,False,False,False,False,False,False,False,False,False,False
1,2004-01-01 01:00:00,16450.0,7.777778,2.222222,5.555556,6.111111,5.555556,6.111111,6.666667,6.666667,...,False,False,False,False,False,False,False,False,False,False
2,2004-01-01 02:00:00,16517.0,7.222222,1.666667,4.444444,5.000000,4.444444,5.555556,5.000000,5.555556,...,False,False,False,False,False,False,False,False,False,False
3,2004-01-01 03:00:00,16873.0,5.000000,-1.111111,2.222222,2.777778,3.888889,3.333333,4.444444,1.111111,...,False,False,False,False,False,False,False,False,False,False
4,2004-01-01 04:00:00,17064.0,3.888889,-1.111111,1.111111,0.555556,4.444444,3.333333,1.666667,-1.111111,...,False,False,False,False,False,False,False,False,False,False


## Split the dataset into train, validation, and test sets

- Training: first 70% of the data
- Validation: next 15% of the data
- Test: last 15% of the data
- get the index positions for the splits

In [86]:
train_start = 0
train_end = int(df.shape[0] * 0.7)

valiation_start = train_end
valiation_end = int(df.shape[0] * 0.85)

test_start = valiation_end
test_end = int(df.shape[0])

y_all =  df["load_kw"]
X_all = df.drop(columns=["load_kw", "time"])

y_train = y_all[train_start:train_end]
X_train = X_all[train_start:train_end]

y_val = y_all[train_end:valiation_end]
X_val = X_all[train_end:valiation_end]

X_test = X_all[valiation_end:test_end]
y_test = y_all[valiation_end:test_end]

y_val.head()



26649    20201.0
26650    17698.0
26651    16322.0
26652    15397.0
26653    14478.0
Name: load_kw, dtype: float64

In [87]:
len(X_test)

5711

## Train the model
- Use the training set to train the model

In [88]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)



LinearRegression()

## Make predictions on all three sets


In [89]:
y_hat_train = model.predict(X_train)
y_hat_val = model.predict(X_val)
y_hat_test = model.predict(X_test)

y_hat_test[:10]

array([17621.02768788, 18248.87948696, 17962.73886791, 16958.93066549,
       16736.14125871, 16920.18405425, 16516.33646077, 16900.90703248,
       16802.14756162, 17171.58993153])

## Compute the MAPE for all three sets


In [90]:
from sklearn.metrics import mean_absolute_percentage_error

mapes = {}
for name, y_hat, y in zip(['train', 'val', 'test'], [y_hat_train, y_hat_val, y_hat_test], [y_train, y_val, y_test]):
    mape = mean_absolute_percentage_error(y, y_hat) * 100
    mapes[name] = mape
    print(f'MAPE for {name} set: {mape:.2f}')

MAPE for train set: 18.94
MAPE for val set: 20.33
MAPE for test set: 16.49


## Plot Predictions vs Actuals

- use plotly to plot the predictions vs actuals for all three sets

In [91]:
# Add the columns prediction_training, prediction_valiation, and prediction_test to the original dataframe

df["prediction_training"] = None
df["prediction_valiation"] = None
df["prediction_test"] = None
df.head()


,time,load_kw,station_1_temp_c,station_2_temp_c,station_3_temp_c,station_4_temp_c,station_5_temp_c,station_6_temp_c,station_7_temp_c,station_8_temp_c,...,month_6,month_7,month_8,month_9,month_10,month_11,month_12,prediction_training,prediction_valiation,prediction_test
0,2004-01-01 00:00:00,16853.0,7.777778,3.333333,6.666667,7.222222,5.555556,6.666667,7.222222,6.111111,...,False,False,False,False,False,False,False,None,None,None
1,2004-01-01 01:00:00,16450.0,7.777778,2.222222,5.555556,6.111111,5.555556,6.111111,6.666667,6.666667,...,False,False,False,False,False,False,False,None,None,None
2,2004-01-01 02:00:00,16517.0,7.222222,1.666667,4.444444,5.000000,4.444444,5.555556,5.000000,5.555556,...,False,False,False,False,False,False,False,None,None,None
3,2004-01-01 03:00:00,16873.0,5.000000,-1.111111,2.222222,2.777778,3.888889,3.333333,4.444444,1.111111,...,False,False,False,False,False,False,False,None,None,None
4,2004-01-01 04:00:00,17064.0,3.888889,-1.111111,1.111111,0.555556,4.444444,3.333333,1.666667,-1.111111,...,False,False,False,False,False,False,False,None,None,None


In [92]:
# add the predictions to the original dataframe
print("length of y_hat_train", len(y_hat_train))
print("length of y_hat_val", len(y_hat_val))
print("length of y_hat_test", len(y_hat_test))

df.loc[0:train_end-1, "prediction_training"] = y_hat_train
df.loc[train_end:valiation_end-1, "prediction_valiation"] = y_hat_val
df.loc[valiation_end:test_end, "prediction_test"] = y_hat_test

df.head()


length of y_hat_train 26649
length of y_hat_val 5710
length of y_hat_test 5711


,time,load_kw,station_1_temp_c,station_2_temp_c,station_3_temp_c,station_4_temp_c,station_5_temp_c,station_6_temp_c,station_7_temp_c,station_8_temp_c,...,month_6,month_7,month_8,month_9,month_10,month_11,month_12,prediction_training,prediction_valiation,prediction_test
0,2004-01-01 00:00:00,16853.0,7.777778,3.333333,6.666667,7.222222,5.555556,6.666667,7.222222,6.111111,...,False,False,False,False,False,False,False,18342.443436,None,None
1,2004-01-01 01:00:00,16450.0,7.777778,2.222222,5.555556,6.111111,5.555556,6.111111,6.666667,6.666667,...,False,False,False,False,False,False,False,18302.182173,None,None
2,2004-01-01 02:00:00,16517.0,7.222222,1.666667,4.444444,5.000000,4.444444,5.555556,5.000000,5.555556,...,False,False,False,False,False,False,False,18551.131673,None,None
3,2004-01-01 03:00:00,16873.0,5.000000,-1.111111,2.222222,2.777778,3.888889,3.333333,4.444444,1.111111,...,False,False,False,False,False,False,False,18488.38927,None,None
4,2004-01-01 04:00:00,17064.0,3.888889,-1.111111,1.111111,0.555556,4.444444,3.333333,1.666667,-1.111111,...,False,False,False,False,False,False,False,19561.088992,None,None


In [93]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# sort the dataframe by time
df.sort_values(by="time", inplace=True)
df.reset_index(drop=True, inplace=True)
df["time"] = pd.to_datetime(df["time"])

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["time"], y=df["load_kw"], mode='lines', name='Load'))
fig.add_trace(go.Scatter(x=df["time"][train_start:train_end], y=df["prediction_training"][train_start:train_end], mode='lines', name='Prediction Train'))
fig.add_trace(go.Scatter(x=df["time"][train_end:valiation_end], y=df["prediction_valiation"][train_end:valiation_end], mode='lines', name='Prediction Validation'))
fig.add_trace(go.Scatter(x=df["time"][valiation_end:test_end], y=df["prediction_test"][valiation_end:test_end], mode='lines', name='Prediction Test'))

fig.show()
